# WO-03 Q1 — the ring as it stands
Existing `resolve_basin_ring` and `basin_ring_signature` at L6, run for each test place and the WO-02 population sample. Read-only; timings are local and labelled as such. Measure, do not optimise.

In [1]:
%matplotlib inline
# Cell 1 — Setup
import sys, time, json, gzip, warnings
warnings.filterwarnings("ignore", message="pandas only supports SQLAlchemy")
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'scripts' / 'edop').exists())
sys.path.insert(0, str(ROOT))
from scripts.shared.db_utils import db_connect
from scripts.edop.areas import engine as E
from scripts.edop.whg import wo03_common as C

conn = db_connect()
C.OUT.mkdir(parents=True, exist_ok=True)
print('commit', C.commit(), '| local database and local timings')
print(len(C.PLACES), 'test places:', ', '.join(C.PLACES))

commit d0bb073 | local database and local timings
10 test places: Timbuktu, Tbilisi, Santa Fe, Kaifeng, Lviv, Poltava, Chaco Canyon, Alexandria, Utqiagvik, Tristan da Cunha


In [2]:
# Cell 2 — Resolve each test place to its L6 basin (one match expected per place)
rows = []
for name, (lat, lon) in C.PLACES.items():
    r = conn.execute(
        "SELECT hybas_id, main_bas, next_down, sub_area, up_area FROM public.basin06 "
        "WHERE ST_Contains(geom, ST_SetSRID(ST_MakePoint(%s, %s), 4326))", (lon, lat)).fetchall()
    rows.append(dict(place=name, lat=lat, lon=lon, n_match=len(r),
                     hybas_id=int(r[0][0]) if r else None,
                     main_bas=int(r[0][1]) if r else None,
                     sub_area_km2=float(r[0][3]) if r else None,
                     up_area_km2=float(r[0][4]) if r else None))
PLACE_DF = pd.DataFrame(rows).set_index('place')
print(PLACE_DF.to_string())

                     lat      lon  n_match      hybas_id      main_bas  sub_area_km2  up_area_km2
place                                                                                            
Timbuktu          16.770   -3.010        1  1.060552e+09  1.060022e+09        3826.4     382644.3
Tbilisi           41.720   44.790        1  2.060617e+09  2.060070e+09       23250.8      23251.9
Santa Fe          35.690 -105.940        1  7.060674e+09  7.060049e+09       14659.0      49334.1
Kaifeng           34.800  114.310        1  4.060676e+09  4.060010e+09       15266.6      15266.9
Lviv              49.840   24.030        1  2.060376e+09  2.060027e+09       16335.7      16336.0
Poltava           49.590   34.550        1  2.060459e+09  2.060008e+09       14511.5      14511.5
Chaco Canyon      36.060 -107.960        1  7.060618e+09  7.060009e+09       11887.9      11888.0
Alexandria        31.200   29.920        1  1.060034e+09  1.060034e+09       16374.7      16374.7
Utqiagvik         71

In [3]:
# Cell 3 — resolve_basin_ring per place: neighbour count, area ratios, corner-only touches, time
RING = {}
out = []
for name, (lat, lon) in C.PLACES.items():
    t0 = time.perf_counter()
    try:
        cdf, rgdf = E.resolve_basin_ring(lat, lon, 6, conn)
    except Exception as ex:
        out.append(dict(place=name, error=f'{type(ex).__name__}: {ex}'))
        continue
    dt = time.perf_counter() - t0
    RING[name] = (cdf, rgdf)
    core_area = PLACE_DF.loc[name, 'sub_area_km2']
    ratio = rgdf['sub_area_km2'] / core_area
    out.append(dict(place=name, core_id=int(cdf['hybas_id'].iloc[0]), core_km2=round(core_area),
                    n_nb=len(rgdf),
                    max_ratio=round(ratio.max(), 2) if len(rgdf) else None,
                    min_ratio=round(ratio.min(), 3) if len(rgdf) else None,
                    n_shared0=int((rgdf['shared_km'] == 0).sum()),
                    shared_km_total=round(rgdf['shared_km'].sum(), 1),
                    resolve_s=round(dt, 3)))
print(pd.DataFrame(out).to_string())
print()
print('max_ratio / min_ratio = largest / smallest neighbour area relative to the core.')
print('n_shared0 = neighbours returned by ST_Touches with zero shared border length (corner contact only).')

              place       core_id  core_km2  n_nb  max_ratio  min_ratio  n_shared0  shared_km_total  resolve_s                                                    error
0          Timbuktu  1.060552e+09    3826.0   5.0       6.52      0.241        0.0            544.9      0.151                                                      NaN
1           Tbilisi  2.060617e+09   23251.0   9.0       0.95      0.128        0.0           1277.9      0.028                                                      NaN
2          Santa Fe  7.060674e+09   14659.0   7.0       1.50      0.016        0.0            775.4      0.017                                                      NaN
3           Kaifeng  4.060676e+09   15267.0   7.0       1.15      0.205        0.0           1085.0      0.027                                                      NaN
4              Lviv  2.060376e+09   16336.0  12.0       1.25      0.009        0.0           1071.5      0.031                                                  

In [4]:
# Cell 4 — The neighbours themselves, one block per place
blocks = []
for name, (cdf, rgdf) in RING.items():
    core_area = PLACE_DF.loc[name, 'sub_area_km2']
    t = rgdf[['hybas_id', 'sub_area_km2', 'shared_km', 'border_bearing', 'centroid_bearing']].copy()
    t['area_ratio'] = (t['sub_area_km2'] / core_area).round(2)
    blocks.append(f"{name}  (core {int(cdf['hybas_id'].iloc[0])}, {core_area:.0f} km2)\n" + t.round(1).to_string(index=False))
print('\n\n'.join(blocks))

Timbuktu  (core 1060551560, 3826 km2)
  hybas_id  sub_area_km2  shared_km  border_bearing  centroid_bearing  area_ratio
1060041510       24966.0      230.3             5.0              46.3         6.5
1060564740       11275.8       61.0           277.4             259.3         3.0
1060551770       10322.8      153.6            91.9             133.2         2.7
1060564960        5734.4       80.7           231.2             214.4         1.5
1060550540         921.4       19.3            72.0              71.6         0.2

Tbilisi  (core 2060616700, 23251 km2)
  hybas_id  sub_area_km2  shared_km  border_bearing  centroid_bearing  area_ratio
2060005120       22011.6      227.5           241.4             240.8         1.0
2060624230       17660.3      181.2            78.1             104.2         0.8
2060005270       13510.8      195.9           326.2             319.1         0.6
2060579910       12151.9       47.7            49.5              44.7         0.5
2060624070       1183

In [5]:
# Cell 5 — basin_ring_signature: time, payload size, neighbour count (include_detail False and True)
# Warm-up first: the first call loads the L6 catalog, so it is timed separately.
name0 = next(iter(RING))
lat0, lon0 = C.PLACES[name0]
t0 = time.perf_counter(); E.single_basin_signature(lat0, lon0, conn, level=6); warm = time.perf_counter() - t0
print(f'warm-up single_basin_signature ({name0}): {warm:.2f}s')

SIG = {}      # (place, detail) -> payload
out = []
for name, (lat, lon) in C.PLACES.items():
    if name not in RING:
        continue
    row = dict(place=name)
    for det in (False, True):
        t0 = time.perf_counter()
        try:
            res = E.basin_ring_signature(lat, lon, conn, level=6, include_detail=det)
        except Exception as ex:
            row[f'error_det{int(det)}'] = f'{type(ex).__name__}: {ex}'
            continue
        dt = time.perf_counter() - t0
        raw = json.dumps(res, default=str).encode()
        SIG[(name, det)] = res
        n = len(res['ring'])
        row.update({f'n_nb': n, f's_det{int(det)}': round(dt, 2), f's_per_basin_det{int(det)}': round(dt / (n + 1), 2),
                    f'KB_det{int(det)}': round(len(raw) / 1024, 1), f'gzKB_det{int(det)}': round(len(gzip.compress(raw)) / 1024, 1)})
    out.append(row)
print(pd.DataFrame(out).to_string())

          place  n_nb  s_det0  s_per_basin_det0  KB_det0  gzKB_det0  s_det1  s_per_basin_det1  KB_det1  gzKB_det1
0      Timbuktu     5    7.82              1.30    133.7        5.2    7.27              1.21    289.4       29.2
1       Tbilisi     9   12.61              1.26    223.2        7.8   13.02              1.30    482.6       47.8
2      Santa Fe     7   10.44              1.31    178.1        6.6   10.64              1.33    388.0       40.3
3       Kaifeng     7    9.65              1.21    178.1        6.1   10.01              1.25    386.2       39.8
4          Lviv    12   16.51              1.27    290.2        8.8   15.96              1.23    626.7       61.0
5       Poltava     4    6.31              1.26    111.2        4.4    6.52              1.30    241.3       25.0
6  Chaco Canyon     7   10.85              1.36    178.5        6.0   10.53              1.32    386.6       38.4
7    Alexandria     5    7.82              1.30    133.4        5.2    6.91             

In [7]:
# Cell 6 — What is in a ring payload: keys, row counts and statuses, centre versus neighbours
res = SIG[(name0, False)]
print('top-level keys      :', list(res))
print('centre keys         :', list(res['center']))
print('ring member keys    :', list(res['ring'][0]))
print('centre row keys     :', list(res['center']['rows'][0]))
print()
out = []
for (name, det), res in SIG.items():
    if det:
        continue
    def stat(sig):
        return pd.Series([r['status'] for r in sig['rows']]).value_counts().to_dict()
    ns = [len(m['signature']['rows']) for m in res['ring']]
    out.append(dict(place=name, centre_rows=len(res['center']['rows']), centre_status=stat(res['center']),
                    nb_rows_min=min(ns) if ns else None, nb_rows_max=max(ns) if ns else None,
                    nb_status_all=pd.Series([r['status'] for m in res['ring'] for r in m['signature']['rows']]).value_counts().to_dict()))
print(pd.DataFrame(out).to_string())

top-level keys      : ['type', 'lat', 'lon', 'level', 'center', 'ring']
centre keys         : ['scope', 'shortfall', 'bands', 'temporal', 'caveats', 'rows']
ring member keys    : ['hybas_id', 'sub_area_km2', 'shared_km', 'border_bearing', 'centroid_bearing', 'neighbor_lat', 'neighbor_lon', 'signature']
centre row keys     : ['variable', 'band', 'method', 'unit_type', 'n_units', 'representative_score', 'representative_raw', 'score_suppressed', 'coverage', 'status', 'coherence', 'modality', 'weight_at_zero', 'caveat', 'year', 'epoch_year', 'units', 'db_col', 'friendly_name']

          place  centre_rows                           centre_status  nb_rows_min  nb_rows_max                                           nb_status_all
0      Timbuktu           52  {'ok': 48, 'outside_active_domain': 4}           52           52                {'ok': 232, 'outside_active_domain': 28}
1       Tbilisi           52  {'ok': 45, 'outside_active_domain': 7}           52           52                {'ok': 

In [8]:
# Cell 7 — Can the position scores be computed once for the set?  Read-only use of engine.attach_values
# attach_values takes any basin_set; here: one call over centre + neighbours, versus one call per basin.
# Also checks that the scores agree between the two routes (they should: percent-rank is over the whole table).
meta = E._CATALOG_CACHE[6]
tbl, view = E._LEVEL_TABLE[6], E._LEVEL_VIEW[6]
def bset(ids):
    return pd.DataFrame({'hybas_id': np.array(ids, dtype='int64'), 'weight': 1.0})

out, diffs = [], []
for name, (cdf, rgdf) in RING.items():
    ids = [int(cdf['hybas_id'].iloc[0])] + [int(h) for h in rgdf['hybas_id']]
    t0 = time.perf_counter(); m_set, c_set, r_set = E.attach_values(bset(ids), meta, conn, tbl, view); t_set = time.perf_counter() - t0
    t0 = time.perf_counter()
    parts = [E.attach_values(bset([h]), meta, conn, tbl, view)[0] for h in ids]
    t_each = time.perf_counter() - t0
    m_each = pd.concat(parts)
    num = [c for c in m_set.columns if pd.api.types.is_numeric_dtype(m_set[c]) and pd.api.types.is_numeric_dtype(m_each[c])]
    a, b = m_set.loc[ids, num].astype(float), m_each.loc[ids, num].astype(float)
    bad = ~np.isclose(a.values, b.values, equal_nan=True, atol=1e-9)
    nonnum = [c for c in m_set.columns if c not in num]
    bad_txt = int((m_set.loc[ids, nonnum].astype(str).values != m_each.loc[ids, nonnum].astype(str).values).sum())
    out.append(dict(place=name, n_basins=len(ids), set_s=round(t_set, 2), each_s=round(t_each, 2),
                    n_numeric_cells=int(bad.size), n_numeric_differ=int(bad.sum()), n_text_cells_differ=bad_txt))
    if bad.any():
        for i, j in zip(*np.where(bad)):
            diffs.append(dict(place=name, hybas_id=ids[i], variable=num[j], set_value=a.iloc[i, j], each_value=b.iloc[i, j]))
print(pd.DataFrame(out).to_string())
print()
print('differing cells (first 40):' if diffs else 'no differing cells')
if diffs:
    print(pd.DataFrame(diffs).head(40).to_string())

          place  n_basins  set_s  each_s  n_numeric_cells  n_numeric_differ  n_text_cells_differ
0      Timbuktu         6   1.55    8.17              270                 0                    0
1       Tbilisi        10   1.36   11.64              450                 0                    0
2      Santa Fe         8   1.19   10.12              360                 0                    0
3       Kaifeng         8   1.30   10.22              360                 0                    0
4          Lviv        13   1.29   15.93              585                 0                    0
5       Poltava         5   1.15    5.73              225                 0                    0
6  Chaco Canyon         8   1.16    9.20              360                 0                    0
7    Alexandria         6   1.15    6.88              270                 0                    0
8     Utqiagvik         4   1.20    4.59              180                 0                    0

no differing cells


In [9]:
# Cell 8 — Weight of a compact neighbour row (ring spec: variable, method, status, score, raw; no detail, no histograms)
KEEP = ['variable', 'method', 'status', 'representative_score', 'representative_raw', 'weight_at_zero']
def compact(sig, keys=KEEP):
    return [{k: r.get(k) for k in keys} for r in sig['rows']]
def nbytes(o):
    return len(json.dumps(o, default=str, separators=(',', ':')).encode())

out = []
for name in RING:
    res = SIG.get((name, False))
    if res is None:
        continue
    centre_full = nbytes(res['center'])
    nb_full = [nbytes(m['signature']) for m in res['ring']]
    nb_cmp = [nbytes(compact(m['signature'])) for m in res['ring']]
    meta_only = [nbytes({k: m[k] for k in ('hybas_id', 'sub_area_km2', 'shared_km', 'border_bearing')}) for m in res['ring']]
    proj = centre_full + sum(c + mo for c, mo in zip(nb_cmp, meta_only))
    raw_proj = json.dumps({'center': res['center'], 'ring': [{'meta': 0, 'rows': compact(m['signature'])} for m in res['ring']]}, default=str).encode()
    out.append(dict(place=name, n_nb=len(nb_full), centre_KB=round(centre_full / 1024, 1),
                    nb_full_KB_mean=round(np.mean(nb_full) / 1024, 1) if nb_full else None,
                    nb_compact_KB_mean=round(np.mean(nb_cmp) / 1024, 1) if nb_cmp else None,
                    projected_total_KB=round(proj / 1024, 1),
                    projected_gz_KB=round(len(gzip.compress(raw_proj)) / 1024, 1)))
print(pd.DataFrame(out).to_string())
print()
print('compact keys:', KEEP)
print('projected_total = full centre + compact neighbour rows + per-neighbour metadata (id, area, shared_km, bearing)')

          place  n_nb  centre_KB  nb_full_KB_mean  nb_compact_KB_mean  projected_total_KB  projected_gz_KB
0      Timbuktu     5       20.1             20.2                 7.7                59.0              4.4
1       Tbilisi     9       20.1             20.2                 7.7                90.3              5.9
2      Santa Fe     7       20.1             20.1                 7.7                74.3              5.3
3       Kaifeng     7       20.1             20.1                 7.7                74.3              4.9
4          Lviv    12       20.2             20.2                 7.7               113.7              6.4
5       Poltava     4       20.1             20.1                 7.6                51.0              3.9
6  Chaco Canyon     7       20.2             20.2                 7.7                74.7              4.8
7    Alexandria     5       20.0             20.1                 7.6                58.6              4.2
8     Utqiagvik     3       20.1     

In [10]:
# Cell 9 — Population sample (WO-02: 306 basins, stratified by climate zone): ring counts and area spread. resolve_basin_ring only.
pts = C.population_points()
out = []
t_start = time.perf_counter()
for p in pts:
    t0 = time.perf_counter()
    try:
        cdf, rgdf = E.resolve_basin_ring(p['lat'], p['lon'], 6, conn)
    except Exception as ex:
        out.append(dict(**p, error=f'{type(ex).__name__}: {ex}')); continue
    dt = time.perf_counter() - t0
    core = conn.execute("SELECT sub_area FROM public.basin06 WHERE hybas_id = %s", (int(cdf['hybas_id'].iloc[0]),)).fetchone()[0]
    ratio = rgdf['sub_area_km2'] / core
    out.append(dict(**p, core_id=int(cdf['hybas_id'].iloc[0]), core_km2=float(core), n_nb=len(rgdf),
                    max_ratio=float(ratio.max()) if len(rgdf) else None, min_ratio=float(ratio.min()) if len(rgdf) else None,
                    n_shared0=int((rgdf['shared_km'] == 0).sum()), resolve_s=dt))
POP = pd.DataFrame(out)
POP.to_csv(C.OUT / 'q1_population_ring.csv', index=False)
print(f'{len(POP)} rows in {time.perf_counter() - t_start:.1f}s | errors: {int(POP["error"].notna().sum()) if "error" in POP else 0}')
if 'error' in POP and POP['error'].notna().any():
    print(POP.loc[POP['error'].notna(), ['hybas_id_l6', 'climate_zone', 'lat', 'lon', 'error']].to_string())
ok = POP[POP['n_nb'].notna()]
print()
print('neighbour count distribution:'); print(ok['n_nb'].astype(int).value_counts().sort_index().to_string())
print()
print(ok[['n_nb', 'max_ratio', 'min_ratio', 'resolve_s']].describe(percentiles=[.1, .5, .9]).round(3).to_string())
print()
print(f'cores with at least one corner-only neighbour (shared_km = 0): {(ok["n_shared0"] > 0).sum()} of {len(ok)}')
print(f'cores with no neighbours: {(ok["n_nb"] == 0).sum()}')

306 rows in 4.7s | errors: 0

neighbour count distribution:
n_nb
0      8
1      2
2      8
3     34
4     54
5     66
6     51
7     35
8     16
9     16
10     7
11     1
12     4
14     1
15     2
18     1

          n_nb  max_ratio  min_ratio  resolve_s
count  306.000    298.000    298.000    306.000
mean     5.490     24.308      6.702      0.013
std      2.475    174.349     62.213      0.007
min      0.000      0.199      0.000      0.004
10%      3.000      1.026      0.013      0.007
50%      5.000      3.640      0.256      0.011
90%      9.000     27.638      2.545      0.020
max     18.000   2904.395   1023.895      0.073

cores with at least one corner-only neighbour (shared_km = 0): 23 of 306
cores with no neighbours: 8


In [11]:
# Cell 10 — Which variables are outside_active_domain or no_data: centre versus neighbours (uses SIG from Cell 5; detail off)
def by_status(sig, status):
    return {r['variable'] for r in sig['rows'] if r['status'] == status}

lines, nd = [], []
for (name, det), res in SIG.items():
    if det:
        continue
    c_out = by_status(res['center'], 'outside_active_domain')
    c_nd = by_status(res['center'], 'no_data')
    diff_rows = []
    for m in res['ring']:
        n_out = by_status(m['signature'], 'outside_active_domain')
        diff_rows.append((m['hybas_id'], sorted(n_out - c_out), sorted(c_out - n_out)))
        for r in m['signature']['rows']:
            if r['status'] == 'no_data':
                nd.append(dict(place=name, hybas_id=m['hybas_id'], variable=r['variable'], band=r['band'],
                               coverage=r['coverage'], caveat=r['caveat']))
    n_same = sum(1 for _, extra, missing in diff_rows if not extra and not missing)
    lines.append(f"{name}: centre outside_active_domain ({len(c_out)}): {sorted(c_out)} | centre no_data: {sorted(c_nd)}")
    lines.append(f"    neighbours with the same outside_active_domain set as the centre: {n_same} of {len(diff_rows)}")
    for hid, extra, missing in diff_rows:
        if extra or missing:
            lines.append(f"    {hid}: outside here, not in centre {extra} | outside in centre, not here {missing}")
lines.append('')
lines.append('no_data rows (neighbours):')
lines.append(pd.DataFrame(nd).to_string(index=False) if nd else 'none')
print('\n'.join(lines))

Timbuktu: centre outside_active_domain (4): ['cropland_extent', 'karst', 'karst_upstream', 'permafrost_extent'] | centre no_data: []
    neighbours with the same outside_active_domain set as the centre: 2 of 5
    1060041510: outside here, not in centre ['cropland_extent_upstream', 'dist_sink', 'pasture_extent', 'pasture_extent_upstream', 'wet_pct_grp1', 'wet_pct_grp1_upstream', 'wet_pct_grp2', 'wet_pct_grp2_upstream'] | outside in centre, not here []
    1060551770: outside here, not in centre ['cropland_extent_upstream'] | outside in centre, not here []
    1060564960: outside here, not in centre [] | outside in centre, not here ['cropland_extent']
Tbilisi: centre outside_active_domain (7): ['karst', 'karst_upstream', 'permafrost_extent', 'wet_pct_grp1', 'wet_pct_grp1_upstream', 'wet_pct_grp2', 'wet_pct_grp2_upstream'] | centre no_data: []
    neighbours with the same outside_active_domain set as the centre: 4 of 9
    2060005120: outside here, not in centre ['dist_sink'] | outside i

In [12]:
# Cell 11 — Cell 7 repeated without pd.concat: attach_values for the set versus one basin at a time, compared cell by cell
# Covers all three frames it returns (scores, class ids, raw values), and lists the columns that are all-NA in a single-basin frame.
meta = E._CATALOG_CACHE[6]
tbl, view = E._LEVEL_TABLE[6], E._LEVEL_VIEW[6]
def bset(ids):
    return pd.DataFrame({'hybas_id': np.array(ids, dtype='int64'), 'weight': 1.0})

def same(va, vb):
    if isinstance(va, (list, tuple, np.ndarray)) or isinstance(vb, (list, tuple, np.ndarray)):
        try:
            return bool(np.allclose(np.asarray(va, dtype=float), np.asarray(vb, dtype=float), equal_nan=True, atol=1e-9))
        except (TypeError, ValueError):
            return list(va) == list(vb)
    if pd.isna(va) and pd.isna(vb):
        return True
    if pd.isna(va) or pd.isna(vb):
        return False
    if isinstance(va, (int, float, np.number)) and isinstance(vb, (int, float, np.number)):
        return bool(np.isclose(float(va), float(vb), atol=1e-9))
    return va == vb

FRAMES = ['scores', 'class_ids', 'raw']
out, diffs, colmismatch, allna = [], [], [], {}
n_single_calls = 0
for name, (cdf, rgdf) in RING.items():
    ids = [int(cdf['hybas_id'].iloc[0])] + [int(h) for h in rgdf['hybas_id']]
    t0 = time.perf_counter(); sets = E.attach_values(bset(ids), meta, conn, tbl, view); t_set = time.perf_counter() - t0
    cells = {f: 0 for f in FRAMES}; nd = {f: 0 for f in FRAMES}
    t0 = time.perf_counter()
    for h in ids:
        ones = E.attach_values(bset([h]), meta, conn, tbl, view)
        n_single_calls += 1
        for label, fs, fo in zip(FRAMES, sets, ones):
            if set(fs.columns) != set(fo.columns):
                colmismatch.append(dict(place=name, hybas_id=h, frame=label,
                                        only_in_set=sorted(set(fs.columns) - set(fo.columns)),
                                        only_in_single=sorted(set(fo.columns) - set(fs.columns))))
            for col in fo.columns:
                if fo[col].isna().all():
                    allna[(label, col)] = allna.get((label, col), 0) + 1
                if col not in fs.columns:
                    continue
                cells[label] += 1
                va, vb = fs.at[h, col], fo.at[h, col]
                if not same(va, vb):
                    nd[label] += 1
                    diffs.append(dict(place=name, hybas_id=h, frame=label, column=col, set_value=va, single_value=vb))
    t_each = time.perf_counter() - t0
    out.append(dict(place=name, n_basins=len(ids), set_s=round(t_set, 2), each_s=round(t_each, 2),
                    **{f'{f}_cells': cells[f] for f in FRAMES}, **{f'{f}_differ': nd[f] for f in FRAMES}))

lines = [pd.DataFrame(out).to_string(), '']
lines.append(f'total single-basin calls: {n_single_calls}')
lines.append(f'differing cells: {len(diffs)}')
if diffs:
    lines.append(pd.DataFrame(diffs).head(40).to_string())
lines.append('')
lines.append(f'column-set mismatches between set call and single call: {len(colmismatch)}')
if colmismatch:
    lines.append(pd.DataFrame(colmismatch).head(20).to_string())
lines.append('')
lines.append('columns that are all-NA in a single-basin frame (count of single-basin calls out of the total above):')
lines.append(pd.DataFrame([dict(frame=k[0], column=k[1], n_calls_all_na=v) for k, v in sorted(allna.items())]).to_string(index=False) if allna else 'none')
print('\n'.join(lines))

          place  n_basins  set_s  each_s  scores_cells  class_ids_cells  raw_cells  scores_differ  class_ids_differ  raw_differ
0      Timbuktu         6   1.26    7.34           330               66        336              0                 0           0
1       Tbilisi        10   1.28   12.34           550              110        560              0                 0           0
2      Santa Fe         8   1.27    9.97           440               88        448              0                 0           0
3       Kaifeng         8   1.21   10.68           440               88        448              0                 0           0
4          Lviv        13   1.62   17.74           715              143        728              0                 0           0
5       Poltava         5   1.16    5.79           275               55        280              0                 0           0
6  Chaco Canyon         8   1.19    9.68           440               88        448              0       

In [13]:
# Cell 12 — Per-neighbour border facts over the 9 test places and the 306-basin sample (read-only; resolver output plus one SQL pass)
# Intersection type and length in metres (is a shared_km of 0 a point contact or a border under 5 m?), border length as a share of the
# core's perimeter, the gap between border_bearing and centroid_bearing, and how far border_mid lies from the shared border.
pairs = []
def collect(source, tag, core_id, rgdf):
    for _, r in rgdf.iterrows():
        pairs.append(dict(source=source, tag=tag, core_id=int(core_id), nb_id=int(r['hybas_id']), nb_km2=float(r['sub_area_km2']),
                          shared_km=float(r['shared_km']), border_bearing=float(r['border_bearing']),
                          centroid_bearing=float(r['centroid_bearing']),
                          mlat=float(r['border_mid_lat']), mlon=float(r['border_mid_lon'])))

for name, (cdf, rgdf) in RING.items():
    collect('test', name, cdf['hybas_id'].iloc[0], rgdf)
pts = C.population_points()
n_resolve_err = 0
for p in pts:
    try:
        cdf, rgdf = E.resolve_basin_ring(p['lat'], p['lon'], 6, conn)
    except Exception:
        n_resolve_err += 1
        continue
    collect('sample', p['hybas_id_l6'], cdf['hybas_id'].iloc[0], rgdf)
PAIRS = pd.DataFrame(pairs)

sql = """
SELECT p.core_id, p.nb_id,
       ST_GeometryType(i.g)                                                        AS inter_type,
       ST_Dimension(i.g)                                                           AS dim,
       CASE WHEN ST_Dimension(i.g) = 1 THEN ST_Length(i.g::geography) ELSE 0 END   AS border_m,
       ST_Perimeter(a.geom::geography)                                             AS core_perimeter_m,
       ST_Distance(ST_SetSRID(ST_MakePoint(p.mlon, p.mlat), 4326)::geography, i.g::geography) AS mid_to_border_m
FROM unnest(%s::bigint[], %s::bigint[], %s::float8[], %s::float8[]) AS p(core_id, nb_id, mlon, mlat)
JOIN public.basin06 a ON a.hybas_id = p.core_id
JOIN public.basin06 b ON b.hybas_id = p.nb_id
CROSS JOIN LATERAL (SELECT ST_Intersection(a.geom, b.geom) AS g) i
"""
uniq = PAIRS.drop_duplicates(['core_id', 'nb_id'])
facts = pd.DataFrame(conn.execute(sql, (uniq['core_id'].tolist(), uniq['nb_id'].tolist(),
                                        uniq['mlon'].tolist(), uniq['mlat'].tolist())).fetchall(),
                     columns=['core_id', 'nb_id', 'inter_type', 'dim', 'border_m', 'core_perimeter_m', 'mid_to_border_m'])
facts['core_id'] = facts['core_id'].astype('int64'); facts['nb_id'] = facts['nb_id'].astype('int64')
PAIRS = PAIRS.merge(facts, on=['core_id', 'nb_id'], how='left')
d = (PAIRS['border_bearing'] - PAIRS['centroid_bearing']).abs()
PAIRS['bearing_gap'] = np.minimum(d, 360 - d)
PAIRS['border_share_of_perimeter'] = PAIRS['border_m'] / PAIRS['core_perimeter_m']
PAIRS.to_csv(C.OUT / 'q1_pairs.csv', index=False)

L = []
L.append(f'pairs: {len(PAIRS)} ({(PAIRS.source == "test").sum()} test places, {(PAIRS.source == "sample").sum()} sample) | unique (core, neighbour): {len(uniq)} | resolver errors: {n_resolve_err} | pairs with no SQL row: {int(PAIRS["dim"].isna().sum())}')
L.append('')
L.append('intersection type (all pairs):')
L.append(PAIRS['inter_type'].value_counts(dropna=False).to_string())
L.append('')
bins = [-1, 0, 5, 50, 500, 5000, 10000, 1e9]
labels = ['0 m (point)', '0-5 m', '5-50 m', '50-500 m', '0.5-5 km', '5-10 km', '>10 km']
L.append('border length in metres, binned:')
L.append(pd.cut(PAIRS['border_m'], bins=bins, labels=labels).value_counts().reindex(labels).to_string())
L.append('')
z = PAIRS[PAIRS['shared_km'] == 0]
L.append(f'pairs with engine shared_km = 0: {len(z)} (in {z.groupby(["source", "core_id"]).ngroups} cores) -> intersection type: {z["inter_type"].value_counts(dropna=False).to_dict()}; border_m: min {z["border_m"].min():.1f}, max {z["border_m"].max():.1f}')
rnd = (PAIRS['shared_km'] - (PAIRS['border_m'] / 1000).round(2)).abs()
L.append(f'engine shared_km versus border_m/1000 rounded to 2 decimals: pairs differing by more than 0.01 km: {int((rnd > 0.01).sum())}')
L.append('')
L.append('border length as a share of the core perimeter (pairs with a line border):')
L.append(PAIRS.loc[PAIRS['dim'] == 1, 'border_share_of_perimeter'].describe(percentiles=[.05, .1, .5, .9]).round(4).to_string())
L.append('')
L.append('bearing gap (degrees, border_bearing vs centroid_bearing):')
L.append(PAIRS['bearing_gap'].describe(percentiles=[.5, .9, .99]).round(1).to_string())
L.append(f'pairs with gap > 90 degrees: {int((PAIRS["bearing_gap"] > 90).sum())} of {len(PAIRS)}')
L.append('')
L.append('distance from border_mid to the shared border (m):')
L.append(PAIRS['mid_to_border_m'].describe(percentiles=[.5, .9, .99]).round(1).to_string())
L.append(f'pairs with border_mid more than 1 km from the border: {int((PAIRS["mid_to_border_m"] > 1000).sum())}; more than 10 km: {int((PAIRS["mid_to_border_m"] > 10000).sum())}')
big = PAIRS['bearing_gap'] > 90
L.append(f'gap > 90 degrees: median distance border_mid to border {PAIRS.loc[big, "mid_to_border_m"].median():.0f} m ({int(big.sum())} pairs); gap <= 90: {PAIRS.loc[~big, "mid_to_border_m"].median():.0f} m')
L.append('')
cols = ['source', 'tag', 'core_id', 'nb_id', 'nb_km2', 'shared_km', 'border_m', 'inter_type', 'border_bearing', 'centroid_bearing', 'bearing_gap', 'mid_to_border_m']
L.append('ten largest bearing gaps:')
L.append(PAIRS.sort_values('bearing_gap', ascending=False)[cols].head(10).round(1).to_string(index=False))
L.append('')
L.append('ten largest border_mid distances:')
L.append(PAIRS.sort_values('mid_to_border_m', ascending=False)[cols].head(10).round(1).to_string(index=False))
print('\n'.join(L))

pairs: 1739 (59 test places, 1680 sample) | unique (core, neighbour): 1739 | resolver errors: 0 | pairs with no SQL row: 0

intersection type (all pairs):
inter_type
ST_MultiLineString       1700
ST_Point                   26
ST_LineString               9
ST_GeometryCollection       4

border length in metres, binned:
border_m
0 m (point)      26
0-5 m             0
5-50 m            0
50-500 m          9
0.5-5 km         82
5-10 km          79
>10 km         1543

pairs with engine shared_km = 0: 26 (in 23 cores) -> intersection type: {'ST_Point': 26}; border_m: min 0.0, max 0.0
engine shared_km versus border_m/1000 rounded to 2 decimals: pairs differing by more than 0.01 km: 0

border length as a share of the core perimeter (pairs with a line border):
count    1713.0000
mean        0.1641
std         0.1336
min         0.0001
5%          0.0083
10%         0.0183
50%         0.1340
90%         0.3502
max         1.0000

bearing gap (degrees, border_bearing vs centroid_bearing):
count

In [14]:
# Cell 13 — The cores with few or no neighbours, the extreme area ratios, and coast/endorheic flags across the sample (uses POP from Cell 9; read-only SQL)
ids = [int(h) for h in POP['core_id']]
flags = pd.DataFrame(conn.execute(
    "SELECT hybas_id, sub_area, up_area, coast, endo, dist_sink, next_down FROM public.basin06 WHERE hybas_id = ANY(%s)", (ids,)).fetchall(),
    columns=['core_id', 'core_km2_db', 'up_area', 'coast', 'endo', 'dist_sink', 'next_down'])
flags['core_id'] = flags['core_id'].astype('int64')
P2 = POP.copy()
P2['core_id'] = P2['core_id'].astype('int64')
P2 = P2.merge(flags, on='core_id', how='left')

L = []
L.append(f'sample cores: {len(P2)} | missing from basin06 lookup: {int(P2["coast"].isna().sum())}')
L.append('value counts, coast: ' + str(P2['coast'].value_counts(dropna=False).to_dict()) + ' | endo: ' + str(P2['endo'].value_counts(dropna=False).to_dict()))
L.append('')
L.append('core area (km2) over the sample:')
L.append(P2['core_km2'].describe(percentiles=[.05, .1, .5, .9]).round(1).to_string())
L.append(f'cores under 100 km2: {int((P2["core_km2"] < 100).sum())}; under 1000 km2: {int((P2["core_km2"] < 1000).sum())}')
L.append('')
L.append('neighbour count by coast flag (rows: coast, columns: n_nb 0, 1-2, 3-5, 6+):')
band = pd.cut(P2['n_nb'], bins=[-1, 0, 2, 5, 99], labels=['0', '1-2', '3-5', '6+'])
L.append(pd.crosstab(P2['coast'], band).to_string())
L.append('')
P2['has_corner'] = P2['n_shared0'] > 0
L.append('cores with a corner-only neighbour (shared_km = 0) by coast flag:')
L.append(pd.crosstab(P2['coast'], P2['has_corner']).to_string())
L.append('')

few = P2[P2['n_nb'] <= 2].sort_values(['n_nb', 'core_id'])
rows = []
for _, r in few.iterrows():
    nn = conn.execute("""
        SELECT n.hybas_id, n.sub_area, ST_Distance(n.geom::geography, a.geom::geography)
        FROM public.basin06 a, public.basin06 n
        WHERE a.hybas_id = %s AND n.hybas_id <> a.hybas_id
        ORDER BY a.geom <-> n.geom LIMIT 1""", (int(r['core_id']),)).fetchone()
    rows.append(dict(core_id=int(r['core_id']), n_nb=int(r['n_nb']), lat=round(r['lat'], 3), lon=round(r['lon'], 3),
                     zone=r['climate_zone'], core_km2=round(r['core_km2'], 1), up_area=r['up_area'], coast=r['coast'], endo=r['endo'],
                     next_down=int(r['next_down']), nearest_id=int(nn[0]), nearest_km2=round(nn[1], 1), nearest_dist_m=round(nn[2], 0)))
L.append(f'cores with 2 or fewer neighbours ({len(few)}); nearest = the nearest other L6 basin, any distance:')
L.append(pd.DataFrame(rows).to_string(index=False))
L.append('')
cols = ['core_id', 'lat', 'lon', 'climate_zone', 'core_km2', 'n_nb', 'max_ratio', 'min_ratio', 'coast', 'endo']
L.append('five largest max_ratio (largest neighbour / core):')
L.append(P2.sort_values('max_ratio', ascending=False)[cols].head(5).round(3).to_string(index=False))
L.append('')
L.append('five largest min_ratio (smallest neighbour / core):')
L.append(P2.sort_values('min_ratio', ascending=False)[cols].head(5).round(3).to_string(index=False))
print('\n'.join(L))

sample cores: 306 | missing from basin06 lookup: 0
value counts, coast: {0: 253, 1: 53} | endo: {0: 250, 2: 39, 1: 17}

core area (km2) over the sample:
count      306.0
mean      8429.0
std       9193.4
min          0.1
5%         248.7
10%        538.3
50%       5259.3
90%      21111.4
max      55897.2
cores under 100 km2: 8; under 1000 km2: 47

neighbour count by coast flag (rows: coast, columns: n_nb 0, 1-2, 3-5, 6+):
n_nb   0  1-2  3-5   6+
coast                  
0      0    5  121  127
1      8    5   33    7

cores with a corner-only neighbour (shared_km = 0) by coast flag:
has_corner  False  True 
coast                   
0             252      1
1              31     22

cores with 2 or fewer neighbours (18); nearest = the nearest other L6 basin, any distance:
   core_id  n_nb     lat      lon  zone  core_km2  up_area  coast  endo  next_down  nearest_id  nearest_km2  nearest_dist_m
2060056060     0  57.597  -13.688    10       0.3      0.3      1     0          0  2060054790 